In [16]:
#Set-up
import requests
from pathlib import Path
import pandas as pd
import zipfile
%pip install remotezip
from remotezip import RemoteZip
import random
import shutil
import h5py
import numpy as np
from matplotlib import pyplot as plt
import tifffile

Note: you may need to restart the kernel to use updated packages.


1) Locate Dataset, Download annotation ZIP file, and Download Anntoation

In [ ]:
#Get dataset
record = "14551288"
url = f"https://zenodo.org/api/records/{record}"
response = requests.get(url)
data = response.json()

#set directory for downloads
download_dir = Path("data")
download_dir.mkdir(exist_ok=True)

#download annotations zip
file = next(f for f in data["files"] if f["key"] == "Annotations_CSV.zip")
file_url = file["links"] = file["links"]["self"]
file_path = download_dir / file["key"]
with requests.get(file_url, stream=True) as r:
    r.raise_for_status()
    with open(file_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)


#Extract annotation file
ann_extract_path = Path("data/annotations")
with zipfile.ZipFile(file_path, "r") as ann_zip:
    ann_zip.extractall(ann_extract_path)

#Download annotation CSV
metadata = pd.read_csv("data/annotations/Annotations_CSV/Semantic annotation/Phase annotation.csv")

2) Partial extract files from cropped cell video zip file, and randomly identify videos to download that meet the required conditions

In [49]:
#Set zip file variable
cell_zip_files = ["Eosinophils_crops.zip"]

#Select number of files to select
num_files = 5

#Identify videos to exclude 
ex_videos = ["Neu7", "Neu11", "Neu17"]

#Set randomizer
random.seed(42)

#Set URL location and partially extract identified video files
selected_files = {}

for cell_name in cell_zip_files:
    file = next(f for f in data["files"] if f["key"] == cell_name)
    file_url = file["links"]["self"]

    with RemoteZip(file_url) as z:

        #Do not include cell videos that have mutliple cells in each cropped video
        files = [f for f in z.infolist() if not f.is_dir() and f.filename.count("_cell") == 1]

        valid_files = []

        for f in files:
            #Get video ID and cell number from filename
            video_name = Path(f.filename).stem
            video_id = video_name.split("_cell")[0]
            cell_num = int(video_name.split("_cell")[1])

            #Exclude specific videos
            if video_id.lower() in [v.lower() for v in ex_videos]:
                continue

            #Find annotations for video ID
            rows = metadata[
                metadata["Video ID"].str.strip().str.lower() == video_id.lower()
                ].reset_index(drop=True)

            #Skip if the cell has no annotation
            if cell_num > len(rows):
                continue

            #Match cell to annotation row
            row = rows.iloc[cell_num - 1]

            #Exract frame boundaries
            frames = {
                "predeath": 1,
                "blebbing_start": row["blebbing (start frame)"],
                "blebbing_end": row["blebbing (end frame)"],
                "disruption_start": row["disruption (start frame)"],
                "disruption_end": row["disruption (end frame)"]
            }

            #Skip videos with missing frame annotations
            if any(pd.isna(frame) for frame in frames.values()):
                continue

            #Skip videos where blebbing starts at frame 1
            if frames["blebbing_start"] <= 1:
                continue

            #Keep videos that meet all conditions
            valid_files.append(f)

        #Print number of eligible files for download
        print(f"\n{cell_name}: {len(valid_files)} eligible files")

        #Randomly select from identified files
        selected = random.sample(valid_files, min(num_files, len(valid_files)))

        selected_files[cell_name] = [f.filename for f in selected]

        #Print file names
        for f in selected:
            print(f.filename, round(f.file_size / 1e6, 2), "MB")



Eosinophils_crops.zip: 7 eligible files
Eosinophils_crops/EOS4_cell2.ims 139.45 MB
Eosinophils_crops/EOS2_cell4.ims 211.45 MB
Eosinophils_crops/EOS5_cell1.ims 226.79 MB
Eosinophils_crops/EOS3_cell4.ims 312.79 MB
Eosinophils_crops/EOS4_cell1.ims 145.38 MB


3) Download identified files and store in data in cropp videos folder

In [50]:
#Download selected files. Note takes approximately 1-2 minute per file download

#Create download path
download_dir = Path("data/cropped_videos")

#Find identified files and download video crop
for cell_name in cell_zip_files:

    file = next(f for f in data["files"] if f["key"] == cell_name)
    file_url = file["links"]["self"]

    # Create folders for each cell type
    cell_type = cell_name.replace("_crops.zip", "")
    output_dir = download_dir / cell_type
    output_dir.mkdir(parents=True, exist_ok=True)

    with RemoteZip(file_url) as z:

        for filename in selected_files[cell_name]:

            output_path = output_dir / filename
            output_path.parent.mkdir(parents=True, exist_ok=True)

            with z.open(filename) as source:
                with open(output_path, "wb") as destination:
                    shutil.copyfileobj(source, destination)

            print("Downloaded:", output_path)



Downloaded: data\cropped_videos\Eosinophils\Eosinophils_crops\EOS4_cell2.ims
Downloaded: data\cropped_videos\Eosinophils\Eosinophils_crops\EOS2_cell4.ims
Downloaded: data\cropped_videos\Eosinophils\Eosinophils_crops\EOS5_cell1.ims
Downloaded: data\cropped_videos\Eosinophils\Eosinophils_crops\EOS3_cell4.ims
Downloaded: data\cropped_videos\Eosinophils\Eosinophils_crops\EOS4_cell1.ims


In [51]:
#store video files
crop_dir = Path("data/cropped_videos")
video_files = list(crop_dir.rglob("*.ims"))

4) Extract predeath, blebbing, and disruption frames, convert to 2D images, and save to frames folder

In [62]:
#Create frames directory
frames_dir = Path("data/frames")
frames_dir.mkdir(parents=True, exist_ok=True)

#set picture pixel crop size
crop_size = 100

#for each video identify its annotation and extract frames to convert to 2d images
for video_path in video_files:

    video_id = video_path.stem.split("_cell")[0]
    cell_num = int(video_path.stem.split("_cell")[1])

    #get video ID
    rows = metadata[
        metadata["Video ID"].str.lower() == video_id.lower()
    ].reset_index(drop=True)

    if cell_num > len(rows):
        print("No matching annotation:", video_path.name)
        continue

    #identify annotation row for that cell
    row = rows.iloc[cell_num - 1]

    #extract frame information 
    frames = {
        "predeath": 1,
        "blebbing_start": row["blebbing (start frame)"],
        "blebbing_end": row["blebbing (end frame)"],
        "disruption_start": row["disruption (start frame)"],
        "disruption_end": row["disruption (end frame)"]
    }

    #Extract channel information
    channel = int(row["Channel"].lower().replace("ch", "").split("/")[0]) - 1

    # Locate Position CSV for the correct video and channel
    folder_name = f"{video_id}_ch{channel + 1}"

    position_files = [
    p for p in ann_extract_path.rglob("*.csv")
    if folder_name.lower() in [part.lower() for part in p.parts]
    and p.stem.lower() == "position"
]


    positions = pd.read_csv(position_files[0])

    # Use first available tracked position
    positions = positions.sort_values("Time")

    #get position coordinates in um
    position = positions.iloc[0]
    x_um = position["Position X"]
    y_um = position["Position Y"]


    #extract frames from video
    with h5py.File(video_path, "r") as video:

        dataset = video["DataSet/ResolutionLevel 0"]
        
        for label, frame_num in frames.items():

            #Skip missing annotations
            if pd.isna(frame_num):
                continue

            #Assumes annotation frames start at 1
            t = int(frame_num) - 1

            frame_path = f"TimePoint {t}/Channel {channel}/Data"

            if frame_path not in dataset:
                print("Missing frame:", video_path.name, frame_path)
                continue

            #Read 3D frame
            frame = dataset[frame_path][:]

            #Convert to 2D image
            image_2d = np.mean(frame, axis=0)

            # Crop center of image
            half = crop_size // 2

            height, width = image_2d.shape

            # Find center coordinates
            x_pixel = width // 2
            y_pixel = height // 2

            # Crop 65 x 65 pixels
            cropped = image_2d[
                y_pixel - half:y_pixel + half + 1,
                x_pixel - half:x_pixel + half + 1
            ]

            #Save image with label
            filename = f"{video_id}_cell{cell_num}_{label}.png"

            filename = f"{video_id}_cell{cell_num}_{label}.tif"

            # Save TIFF (preserves original intensity values)
            tifffile.imwrite(
                frames_dir / f"{filename}.tif",
                cropped
            )

            # Save PNG (for easy viewing)
            plt.imsave(
                frames_dir / f"{filename}.png",
                cropped,
                cmap="gray"
            )

            print("Saved:", filename)    

Saved: EOS2_cell4_predeath.tif
Saved: EOS2_cell4_blebbing_start.tif
Saved: EOS2_cell4_blebbing_end.tif
Saved: EOS2_cell4_disruption_start.tif
Saved: EOS2_cell4_disruption_end.tif
Saved: EOS3_cell4_predeath.tif
Saved: EOS3_cell4_blebbing_start.tif
Saved: EOS3_cell4_blebbing_end.tif
Saved: EOS3_cell4_disruption_start.tif
Saved: EOS3_cell4_disruption_end.tif
Saved: EOS4_cell1_predeath.tif
Saved: EOS4_cell1_blebbing_start.tif
Saved: EOS4_cell1_blebbing_end.tif
Saved: EOS4_cell1_disruption_start.tif
Saved: EOS4_cell1_disruption_end.tif
Saved: EOS4_cell2_predeath.tif
Saved: EOS4_cell2_blebbing_start.tif
Saved: EOS4_cell2_blebbing_end.tif
Saved: EOS4_cell2_disruption_start.tif
Saved: EOS4_cell2_disruption_end.tif
Saved: EOS5_cell1_predeath.tif
Saved: EOS5_cell1_blebbing_start.tif
Saved: EOS5_cell1_blebbing_end.tif
Saved: EOS5_cell1_disruption_start.tif
Saved: EOS5_cell1_disruption_end.tif
